In [ ]:

import os, time, requests, unicodedata, pandas as pd, csv, sys, signal, random

# ---------- CONFIG ----------
YELP_API_KEY = os.getenv("YELP_API_KEY", "").strip() or ""
if not YELP_API_KEY or "PASTE" in YELP_API_KEY:
    raise SystemExit("❌ Please set your Yelp API key (YELP_API_KEY).")

HEADERS = {"Authorization": f"Bearer {YELP_API_KEY}"}
API_URL = "https://api.yelp.com/v3/businesses/search"

TARGET_ROWS = 20000
LIMIT = 50
MAX_OFFSET = 950
RADIUS = 6000        # 6 km to avoid 400-offset errors
SLEEP = 0.25
CHECKPOINT_EVERY = 500
OUTPUT = "yelp_bigcities_densegrid_progress.csv"

# ---------- COORDINATE GRIDS ----------
# Generated roughly every ~6 km around each city center.
def city_grid(lat, lon, step_lat=0.05, step_lon=0.06, n=3):
    """create (2n+1)^2 grid around a city center"""
    pts=[]
    for i in range(-n,n+1):
        for j in range(-n,n+1):
            pts.append((lat+i*step_lat, lon+j*step_lon))
    return pts

BIG_CITY_GRIDS = {
    "New York":      city_grid(40.73,-73.98, 0.05,0.06, n=4),   # ~80 pts
    "Los Angeles":   city_grid(34.05,-118.24,0.05,0.07, n=3),   # ~49 pts
    "Chicago":       city_grid(41.88,-87.63, 0.04,0.05, n=3),
    "Houston":       city_grid(29.76,-95.37, 0.05,0.06, n=3),
    "Phoenix":       city_grid(33.45,-112.07,0.05,0.06, n=3),
    "Philadelphia":  city_grid(39.95,-75.16, 0.04,0.05, n=2),
    "San Diego":     city_grid(32.72,-117.16,0.04,0.05, n=2),
    "Dallas":        city_grid(32.78,-96.80, 0.05,0.06, n=2),
    "San Francisco": city_grid(37.77,-122.42,0.03,0.04, n=2),
    "Austin":        city_grid(30.27,-97.74, 0.05,0.05, n=2),
    "Miami":         city_grid(25.76,-80.19, 0.04,0.05, n=2),
    "Atlanta":       city_grid(33.75,-84.39, 0.05,0.05, n=2),
    "Boston":        city_grid(42.36,-71.06, 0.04,0.05, n=2),
    "Seattle":       city_grid(47.61,-122.33,0.04,0.05, n=2),
    "Denver":        city_grid(39.74,-104.99,0.05,0.06, n=2)
}

# Flatten to shuffled list of all (city, lat, lon)
ALL_POINTS = [(city,lat,lon) for city,pts in BIG_CITY_GRIDS.items() for (lat,lon) in pts]
random.shuffle(ALL_POINTS)

# ---------- HELPERS ----------
def strip_accents(s): return ''.join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
def norm(s): return strip_accents((s or "").lower().strip())

CUISINE_MAP = {
    "italian":"italian","pizza":"pizza","mexican":"mexican","chinese":"chinese",
    "japanese":"japanese","sushi":"japanese","thai":"thai","burgers":"burgers",
    "bbq":"bbq","seafood":"seafood","french":"french","indian":"indian",
    "mediterranean":"mediterranean","greek":"greek","american":"american",
    "vegan":"vegan","vegetarian":"vegetarian","sandwiches":"sandwiches",
    "cafes":"cafe","breakfast_brunch":"breakfast","latin":"latin","tapas":"spanish",
}
def cuisine_tags(cats):
    out=set()
    for c in cats or []:
        alias=norm(c.get("alias",""))
        if alias in CUISINE_MAP: out.add(CUISINE_MAP[alias])
    return "|".join(sorted(out)) if out else None
def row_key(biz):
    loc=biz.get("location",{})
    return (biz.get("id"), norm(biz.get("name","")), norm(loc.get("address1","")), norm(loc.get("city","")))

# ---------- LOAD EXISTING ----------
rows, seen = [], set()
if os.path.exists(OUTPUT):
    print(f"🔁 Resuming from {OUTPUT}")
    df=pd.read_csv(OUTPUT)
    for _,r in df.iterrows():
        seen.add((r["restaurant_id"], norm(r["name"]), norm(str(r["address"])), norm(str(r["city"]))))
    rows=df.to_dict("records")
    print(f"   → Loaded {len(rows):,} entries")

def save_checkpoint():
    pd.DataFrame(rows).to_csv(OUTPUT,index=False,quoting=csv.QUOTE_NONNUMERIC)
    print(f"Saved progress ({len(rows):,} rows)")

def handle_interrupt(sig,frame):
    print("\n Interrupted. Saving progress...")
    save_checkpoint(); sys.exit(0)
signal.signal(signal.SIGINT, handle_interrupt)

# ---------- FETCH ----------
def fetch_point(city, lat, lon):
    total=0
    for offset in range(0, MAX_OFFSET+1, LIMIT):
        params={"term":"restaurants","latitude":lat,"longitude":lon,"radius":RADIUS,
                "limit":LIMIT,"offset":offset}
        r=requests.get(API_URL,headers=HEADERS,params=params,timeout=30)
        if r.status_code==400: break
        if r.status_code==429:
            print("   ⏳ rate-limit, sleeping 5 s"); time.sleep(5); continue
        r.raise_for_status()
        bizs=r.json().get("businesses",[])
        if not bizs: break
        for biz in bizs:
            key=row_key(biz)
            if key in seen: continue
            seen.add(key)
            loc,co=biz.get("location",{}),biz.get("coordinates",{})
            rows.append({
                "restaurant_id":biz.get("id"),"name":biz.get("name"),
                "cuisines":cuisine_tags(biz.get("categories")),
                "price":biz.get("price"),"rating":biz.get("rating"),
                "review_count":biz.get("review_count"),"phone":biz.get("phone"),
                "url":biz.get("url"),"address":loc.get("address1"),
                "city":loc.get("city"),"state":loc.get("state"),
                "zip_code":loc.get("zip_code"),"country":loc.get("country"),
                "latitude":co.get("latitude"),"longitude":co.get("longitude"),
                "source":"yelp"})
            if len(rows)%CHECKPOINT_EVERY==0: save_checkpoint()
        total+=len(bizs)
        if len(rows)>=TARGET_ROWS: return total
        time.sleep(SLEEP)
    return total

# ---------- MAIN LOOP ----------
for idx,(city,lat,lon) in enumerate(ALL_POINTS,1):
    if len(rows)>=TARGET_ROWS: break
    print(f"📍 {idx}/{len(ALL_POINTS)} {city} ({lat:.2f},{lon:.2f}) — total {len(rows):,}")
    added=fetch_point(city,lat,lon)
    print(f"   ↳ added {added} — total {len(rows):,}")
    if len(rows)>=TARGET_ROWS: break
    time.sleep(0.3)

save_checkpoint()
print(f"Finished! {len(rows):,} restaurants saved to {OUTPUT}")


In [ ]:

import os, time, requests, unicodedata, pandas as pd, csv, sys, signal, random

# ---------- CONFIG ----------
YELP_API_KEY = os.getenv("YELP_API_KEY", "").strip() or ""
if not YELP_API_KEY or "PASTE" in YELP_API_KEY:
    raise SystemExit("❌ Please set your Yelp API key (YELP_API_KEY).")

HEADERS = {"Authorization": f"Bearer {YELP_API_KEY}"}
API_URL = "https://api.yelp.com/v3/businesses/search"

APPEND_ROWS = 40000          # collect this many NEW rows
LIMIT = 50
MAX_OFFSET = 950
RADIUS = 6000
SLEEP = 0.25
CHECKPOINT_EVERY = 500
OUTPUT = "yelp.csv"

# ---------- BASE GRID (shifted to new areas) ----------
def city_grid(lat, lon, step_lat=0.05, step_lon=0.06, n=5, offset_shift=0.025):
    """Generate shifted grid so we hit fresh coordinates."""
    pts=[]
    for i in range(-n,n+1):
        for j in range(-n,n+1):
            pts.append((lat+i*step_lat+offset_shift, lon+j*step_lon+offset_shift))
    return pts

BIG_CITY_GRIDS = {
    "New York":      city_grid(40.73,-73.98,n=4,offset_shift=0.03),
    "Los Angeles":   city_grid(34.05,-118.24,n=3,offset_shift=0.02),
    "Chicago":       city_grid(41.88,-87.63,n=3,offset_shift=0.02),
    "Houston":       city_grid(29.76,-95.37,n=3,offset_shift=0.03),
    "Phoenix":       city_grid(33.45,-112.07,n=3,offset_shift=0.03),
    "Philadelphia":  city_grid(39.95,-75.16,n=2,offset_shift=0.02),
    "San Diego":     city_grid(32.72,-117.16,n=2,offset_shift=0.02),
    "Dallas":        city_grid(32.78,-96.80,n=2,offset_shift=0.02),
    "San Francisco": city_grid(37.77,-122.42,n=2,offset_shift=0.015),
    "Austin":        city_grid(30.27,-97.74,n=2,offset_shift=0.02),
    "Miami":         city_grid(25.76,-80.19,n=2,offset_shift=0.02),
    "Atlanta":       city_grid(33.75,-84.39,n=2,offset_shift=0.02),
    "Boston":        city_grid(42.36,-71.06,n=2,offset_shift=0.02),
    "Seattle":       city_grid(47.61,-122.33,n=2,offset_shift=0.02),
    "Denver":        city_grid(39.74,-104.99,n=2,offset_shift=0.02)
}

ALL_POINTS = [(city,lat,lon) for city,pts in BIG_CITY_GRIDS.items() for (lat,lon) in pts]
random.shuffle(ALL_POINTS)

# ---------- HELPERS ----------
def strip_accents(s): return ''.join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
def norm(s): return strip_accents((s or "").lower().strip())

CUISINE_MAP = {
    "italian":"italian","pizza":"pizza","mexican":"mexican","chinese":"chinese",
    "japanese":"japanese","sushi":"japanese","thai":"thai","burgers":"burgers",
    "bbq":"bbq","seafood":"seafood","french":"french","indian":"indian",
    "mediterranean":"mediterranean","greek":"greek","american":"american",
    "vegan":"vegan","vegetarian":"vegetarian","sandwiches":"sandwiches",
    "cafes":"cafe","breakfast_brunch":"breakfast","latin":"latin","tapas":"spanish",
}
def cuisine_tags(cats):
    out=set()
    for c in cats or []:
        alias=norm(c.get("alias",""))
        if alias in CUISINE_MAP: out.add(CUISINE_MAP[alias])
    return "|".join(sorted(out)) if out else None

def row_key(biz):
    loc=biz.get("location",{})
    return (biz.get("id"), norm(biz.get("name","")), norm(loc.get("address1","")), norm(loc.get("city","")))

# ---------- LOAD EXISTING ----------
rows = []
seen = set()
if os.path.exists(OUTPUT):
    print(f"🔁 Resuming & appending to {OUTPUT}")
    df = pd.read_csv(OUTPUT)
    for _,r in df.iterrows():
        seen.add((r["restaurant_id"], norm(r["name"]), norm(str(r["address"])), norm(str(r["city"]))))
    rows = df.to_dict("records")
    print(f"   → Loaded {len(rows):,} existing rows.")

base_count = len(rows)

def save_checkpoint():
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT, index=False, quoting=csv.QUOTE_NONNUMERIC)
    print(f"💾 Saved progress ({len(rows):,} total rows).")

def handle_interrupt(sig, frame):
    print("\n⚠️ Interrupted. Saving progress...")
    save_checkpoint()
    sys.exit(0)
signal.signal(signal.SIGINT, handle_interrupt)

# ---------- FETCH ----------
def fetch_point(city, lat, lon):
    total = 0
    for offset in range(0, MAX_OFFSET+1, LIMIT):
        params = {
            "term": "restaurants",
            "latitude": lat,
            "longitude": lon,
            "radius": RADIUS,
            "limit": LIMIT,
            "offset": offset
        }
        r = requests.get(API_URL, headers=HEADERS, params=params, timeout=30)
        if r.status_code == 400:
            break
        if r.status_code == 429:
            print("   ⏳ Rate limit, sleeping 5 s"); time.sleep(5); continue
        r.raise_for_status()
        bizs = r.json().get("businesses", [])
        if not bizs: break

        for biz in bizs:
            key = row_key(biz)
            if key in seen: continue
            seen.add(key)
            loc, co = biz.get("location", {}), biz.get("coordinates", {})
            rows.append({
                "restaurant_id": biz.get("id"),
                "name": biz.get("name"),
                "cuisines": cuisine_tags(biz.get("categories")),
                "price": biz.get("price"),
                "rating": biz.get("rating"),
                "review_count": biz.get("review_count"),
                "phone": biz.get("phone"),
                "url": biz.get("url"),
                "address": loc.get("address1"),
                "city": loc.get("city"),
                "state": loc.get("state"),
                "zip_code": loc.get("zip_code"),
                "country": loc.get("country"),
                "latitude": co.get("latitude"),
                "longitude": co.get("longitude"),
                "source": "yelp"
            })
            if (len(rows) - base_count) % CHECKPOINT_EVERY == 0:
                save_checkpoint()
        total += len(bizs)
        if (len(rows) - base_count) >= APPEND_ROWS: return total
        time.sleep(SLEEP)
    return total

# ---------- MAIN ----------
for idx,(city,lat,lon) in enumerate(ALL_POINTS,1):
    if (len(rows) - base_count) >= APPEND_ROWS: break
    print(f"📍 {idx}/{len(ALL_POINTS)} {city} ({lat:.2f},{lon:.2f}) — added {(len(rows)-base_count):,}")
    added = fetch_point(city, lat, lon)
    print(f"   ↳ added {added} — total {(len(rows)-base_count):,} new, {len(rows):,} overall")
    if (len(rows) - base_count) >= APPEND_ROWS: break
    time.sleep(0.3)

save_checkpoint()
print(f"✅ Done! {len(rows):,} total rows in {OUTPUT}")


In [ ]:

import os, time, requests, unicodedata, pandas as pd, csv, sys, signal, random

# ---------- CONFIG ----------
YELP_API_KEY = os.getenv("YELP_API_KEY", "").strip() or ""
if not YELP_API_KEY or "PASTE" in YELP_API_KEY:
    raise SystemExit("❌ Please set your Yelp API key.")

HEADERS = {"Authorization": f"Bearer {YELP_API_KEY}"}
API_URL = "https://api.yelp.com/v3/businesses/search"

TARGET_TOTAL = 60000        # stop when file reaches this many rows
LIMIT = 50
MAX_OFFSET = 950
RADIUS = 6000
SLEEP = 0.25
CHECKPOINT_EVERY = 500
OUTPUT = "yelp.csv"

# ---------- HELPER GRID ----------
def city_grid(lat, lon, step_lat=0.05, step_lon=0.06, n=3, offset_shift=0.02):
    pts=[]
    for i in range(-n,n+1):
        for j in range(-n,n+1):
            pts.append((lat+i*step_lat+offset_shift, lon+j*step_lon+offset_shift))
    return pts

# ---------- MID-TIER CITY GRIDS ----------
MID_CITY_GRIDS = {
    "Nashville":     city_grid(36.16,-86.78,n=2),
    "Charlotte":     city_grid(35.23,-80.84,n=2),
    "Minneapolis":   city_grid(44.98,-93.26,n=2),
    "Tampa":         city_grid(27.95,-82.46,n=2),
    "St. Louis":     city_grid(38.63,-90.20,n=2),
    "Las Vegas":     city_grid(36.17,-115.14,n=2),
    "Orlando":       city_grid(28.54,-81.38,n=2),
    "Indianapolis":  city_grid(39.77,-86.16,n=2),
    "Kansas City":   city_grid(39.10,-94.58,n=2),
    "Cleveland":     city_grid(41.50,-81.69,n=2),
    "Columbus":      city_grid(39.96,-83.00,n=2),
    "Portland":      city_grid(45.52,-122.68,n=2),
    "San Jose":      city_grid(37.34,-121.89,n=2),
    "Sacramento":    city_grid(38.58,-121.49,n=2),
    "Baltimore":     city_grid(39.29,-76.61,n=2),
    "Pittsburgh":    city_grid(40.44,-79.99,n=2),
    "Cincinnati":    city_grid(39.10,-84.51,n=2),
    "Raleigh":       city_grid(35.78,-78.64,n=2),
    "Salt Lake City":city_grid(40.76,-111.89,n=2),
    "Milwaukee":     city_grid(43.04,-87.91,n=2),
    "New Orleans":   city_grid(29.95,-90.07,n=2),
    "Oklahoma City": city_grid(35.47,-97.52,n=2),
    "Memphis":       city_grid(35.15,-90.05,n=2),
    "Louisville":    city_grid(38.25,-85.76,n=2),
    "Richmond":      city_grid(37.54,-77.44,n=2),
    "Hartford":      city_grid(41.77,-72.68,n=2),
    "Providence":    city_grid(41.83,-71.41,n=2),
    "Honolulu":      city_grid(21.31,-157.86,n=2),
    "Des Moines":    city_grid(41.59,-93.62,n=2)
}

ALL_POINTS = [(city,lat,lon) for city,pts in MID_CITY_GRIDS.items() for (lat,lon) in pts]
random.shuffle(ALL_POINTS)

# ---------- HELPERS ----------
def strip_accents(s): return ''.join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
def norm(s): return strip_accents((s or "").lower().strip())

CUISINE_MAP = {
    "italian":"italian","pizza":"pizza","mexican":"mexican","chinese":"chinese",
    "japanese":"japanese","sushi":"japanese","thai":"thai","burgers":"burgers",
    "bbq":"bbq","seafood":"seafood","french":"french","indian":"indian",
    "mediterranean":"mediterranean","greek":"greek","american":"american",
    "vegan":"vegan","vegetarian":"vegetarian","sandwiches":"sandwiches",
    "cafes":"cafe","breakfast_brunch":"breakfast","latin":"latin","tapas":"spanish",
}
def cuisine_tags(cats):
    out=set()
    for c in cats or []:
        alias=norm(c.get("alias",""))
        if alias in CUISINE_MAP: out.add(CUISINE_MAP[alias])
    return "|".join(sorted(out)) if out else None
def row_key(biz):
    loc=biz.get("location",{})
    return (biz.get("id"), norm(biz.get("name","")), norm(loc.get("address1","")), norm(loc.get("city","")))

# ---------- LOAD EXISTING ----------
rows=[]
seen=set()
if os.path.exists(OUTPUT):
    print(f"🔁 Resuming & appending to {OUTPUT}")
    df=pd.read_csv(OUTPUT)
    for _,r in df.iterrows():
        seen.add((r["restaurant_id"], norm(r["name"]), norm(str(r["address"])), norm(str(r["city"]))))
    rows=df.to_dict("records")
    print(f"   → Loaded {len(rows):,} existing rows.")
base_count=len(rows)

def save_checkpoint():
    pd.DataFrame(rows).to_csv(OUTPUT,index=False,quoting=csv.QUOTE_NONNUMERIC)
    print(f"💾 Saved progress ({len(rows):,} total rows).")

def handle_interrupt(sig,frame):
    print("\n⚠️ Interrupted, saving progress...")
    save_checkpoint(); sys.exit(0)
signal.signal(signal.SIGINT, handle_interrupt)

# ---------- FETCH ----------
def fetch_point(city,lat,lon):
    total=0
    for offset in range(0,MAX_OFFSET+1,LIMIT):
        params={"term":"restaurants","latitude":lat,"longitude":lon,
                "radius":RADIUS,"limit":LIMIT,"offset":offset}
        r=requests.get(API_URL,headers=HEADERS,params=params,timeout=30)
        if r.status_code==400: break
        if r.status_code==429:
            print("   ⏳ Rate limit, sleeping 5 s"); time.sleep(5); continue
        r.raise_for_status()
        bizs=r.json().get("businesses",[])
        if not bizs: break
        for biz in bizs:
            key=row_key(biz)
            if key in seen: continue
            seen.add(key)
            loc,co=biz.get("location",{}),biz.get("coordinates",{})
            rows.append({
                "restaurant_id":biz.get("id"),"name":biz.get("name"),
                "cuisines":cuisine_tags(biz.get("categories")),
                "price":biz.get("price"),"rating":biz.get("rating"),
                "review_count":biz.get("review_count"),"phone":biz.get("phone"),
                "url":biz.get("url"),"address":loc.get("address1"),
                "city":loc.get("city"),"state":loc.get("state"),
                "zip_code":loc.get("zip_code"),"country":loc.get("country"),
                "latitude":co.get("latitude"),"longitude":co.get("longitude"),
                "source":"yelp"})
            if (len(rows)%CHECKPOINT_EVERY)==0: save_checkpoint()
        total+=len(bizs)
        if len(rows)>=TARGET_TOTAL: return total
        time.sleep(SLEEP)
    return total

# ---------- MAIN ----------
for idx,(city,lat,lon) in enumerate(ALL_POINTS,1):
    if len(rows)>=TARGET_TOTAL: break
    print(f"📍 {idx}/{len(ALL_POINTS)} {city} ({lat:.2f},{lon:.2f}) — total {len(rows):,}")
    added=fetch_point(city,lat,lon)
    print(f"   ↳ added {added} — total {len(rows):,}")
    if len(rows)>=TARGET_TOTAL: break
    time.sleep(0.3)

save_checkpoint()
print(f"✅ Done! {len(rows):,} total rows in {OUTPUT}")